In [25]:
import einops
from tqdm.notebook import tqdm
from torchsummary import summary

import torch
from torch import nn
import torchvision
import torch.optim as optim
from torchvision.transforms import Compose, Resize, ToTensor, Normalize, RandomHorizontalFlip, RandomCrop


In [2]:
torch.__version__, torchvision.__version__

('2.7.0.dev20250212', '0.22.0.dev20250212')

In [3]:
!jupyter nbextension enable --py widgetsnbextension

usage: jupyter [-h] [--version] [--config-dir] [--data-dir] [--runtime-dir]
               [--paths] [--json] [--debug]
               [subcommand]

Jupyter: Interactive Computing

positional arguments:
  subcommand     the subcommand to launch

options:
  -h, --help     show this help message and exit
  --version      show the versions of core jupyter packages and exit
  --config-dir   show Jupyter config dir
  --data-dir     show Jupyter data dir
  --runtime-dir  show Jupyter runtime dir
  --paths        show all Jupyter paths. Add --json for machine-readable
                 format.
  --json         output paths as machine-readable json
  --debug        output debug information about paths

Available subcommands: dejavu events execute kernel kernelspec lab
labextension labhub migrate nbconvert notebook run server troubleshoot trust

Jupyter command `jupyter-nbextension` not found.


In [32]:
device = 'cpu'

patch_size = 16         # Patch size (P) = 16
latent_size = 768       # Latent vector (D). ViT-Base uses 768
n_channels = 3          # Number of channels for input images
num_heads = 12          # ViT-Base uses 12 heads
num_encoders = 12       # ViT-Base uses 12 encoder layers
dropout = 0.1           # Dropout = 0.1 is used with ViT-Base & ImageNet-21k
num_classes = 10        # Number of classes in CIFAR10 dataset
size = 224              # Size used for training = 224

epochs = 30             # Number of epochs
base_lr = 10e-3         # Base LR
weight_decay = 0.03     # Weight decay for ViT-Base (on ImageNet-21k)
batch_size = 1


In [33]:
class InputEmbedding(nn.Module):
    def __init__(self, patch_size=patch_size, n_channels=n_channels, device=device, latent_size=latent_size, batch_size=batch_size):
        super(InputEmbedding, self).__init__()
        self.latent_size = latent_size
        self.patch_size = patch_size
        self.n_channels = n_channels
        self.device = device
        self.batch_size = batch_size
        self.input_size = self.patch_size*self.patch_size*self.n_channels

        # Linear projection
        self.linearProjection = nn.Linear(self.input_size, self.latent_size)

        # Class token
        self.class_token = nn.Parameter(torch.randn(self.batch_size, 1, self.latent_size)).to(self.device)

        # Positional embedding
        self.pos_embedding = nn.Parameter(torch.randn(self.batch_size, 1, self.latent_size)).to(self.device)

    def forward(self, input_data):
        input_data = input_data.to(self.device)

        # Patchify input image
        patches = einops.rearrange(
            input_data, 'b c (h h1) (w w1) -> b (h w) (h1 w1 c)', h1=self.patch_size, w1=self.patch_size)

        #print(input_data.size())
        #print(patches.size())
        
        linear_projection = self.linearProjection(patches).to(self.device)
        b, n, _ = linear_projection.shape

        linear_projection = torch.cat((self.class_token, linear_projection), dim=1)
        pos_embed = einops.repeat(self.pos_embedding, 'b 1 d -> b m d', m=n+1)
    
        linear_projection += pos_embed

        return linear_projection


In [42]:
test_input = torch.randn((1, 3, 224, 224))
test_class = InputEmbedding().to(device)
embed_test = test_class(test_input)

In [43]:
class EncoderBlock(nn.Module):
    def __init__(self, latent_size=latent_size, num_heads=num_heads, device=device, dropout=dropout):
        super(EncoderBlock, self).__init__()

        self.latent_size = latent_size
        self.num_heads = num_heads
        self.device = device
        self.dropout = dropout

        # Normalization layer
        self.norm = nn.LayerNorm(self.latent_size)

        self.multihead = nn.MultiheadAttention(
            self.latent_size, self.num_heads, dropout=self.dropout
        )

        self.enc_MLP = nn.Sequential(
            nn.Linear(self.latent_size, self.latent_size*4),
            nn.GELU(),
            nn.Dropout(self.dropout),
            nn.Linear(self.latent_size*4, self.latent_size),
            nn.Dropout(self.dropout)
        )

    def forward(self, embedded_patches):
        firstnorm_out = self.norm(embedded_patches)
        attention_out = self.multihead(firstnorm_out, firstnorm_out, firstnorm_out)[0]

        # first residual connection
        first_added = attention_out + embedded_patches

        secondnorm_out = self.norm(first_added)
        ff_out = self.enc_MLP(secondnorm_out)

        return ff_out + first_added

In [44]:
test_encoder = EncoderBlock().to(device)
test_encoder(embed_test)

tensor([[[-1.8480,  0.0441, -1.3858,  ..., -0.3164, -0.0986, -0.6463],
         [-1.6824, -0.0700, -2.9480,  ...,  0.9888, -0.4758, -1.2236],
         [-1.5174,  1.6600, -2.4124,  ...,  0.3452,  0.1405, -2.4188],
         ...,
         [-1.8640,  2.7340, -3.5582,  ...,  0.5739, -0.1416, -0.8718],
         [-2.1268,  1.7458, -2.6545,  ...,  0.2838, -0.5538, -0.9018],
         [-1.8400,  1.4069, -1.7452,  ...,  0.6450,  0.3824, -1.1085]]],
       grad_fn=<AddBackward0>)

In [45]:
class Vit(nn.Module):
    def __init__(
            self, 
            num_encoders=num_encoders,
            latent_size=latent_size, 
            device=device,
            num_classes=num_classes,
            dropout=dropout
        ):
        super(Vit, self).__init__()

        self.num_encoders = num_encoders
        self.latent_size = latent_size
        self.device = device
        self.num_classes = num_classes
        self.dropout = dropout

        self.embedding = InputEmbedding()

        # create the stack of encoders
        self.encStack = nn.ModuleList([EncoderBlock() for i in range(self.num_encoders)])

        self.MLP_head = nn.Sequential(
            nn.LayerNorm(self.latent_size),
            nn.Linear(self.latent_size, self.latent_size),
            nn.Linear(self.latent_size, self.num_classes)
        )
    
    def forward(self, test_input):
        enc_output = self.embedding(test_input)

        for enc_layer in self.encStack:
            enc_output = enc_layer(enc_output)
        
        cls_token_embed = enc_output[:,0]

        return self.MLP_head(cls_token_embed)


In [46]:
# Test Vit

model = Vit().to(device)
vit_output = model(test_input)

print(vit_output)
print(vit_output.size())
test_input.shape

tensor([[ 0.2885,  0.1181, -0.2717, -0.3272,  0.0459,  0.0945, -0.5434, -0.0108,
         -0.3756, -0.4106]], grad_fn=<AddmmBackward0>)
torch.Size([1, 10])


torch.Size([1, 3, 224, 224])